# Unified Data Model: Gateway Logs + Pipeline Logs

This notebook creates views and tables to connect:
- **Gateway logs**: Databricks pipeline events (streaming metrics, flow progress)
- **Pipeline logs**: SQL Server Change Tracking sync metrics (rows written, table duration)

## Join Strategy

| Gateway Logs | Pipeline Logs | Join Logic |
|--------------|---------------|------------|
| `update_id` | `run_id` | Direct match |
| `flow_name` (e.g., `gshen_sqlserver_ct.database_dbo_loadtest_pipeline.table_207`) | `table_name` (e.g., `table_1`) | Regex extract table number |


In [ ]:
%sql
-- ============================================================
-- CONFIGURATION: Set your catalog and schema names
-- ============================================================

SET VAR catalog_name = 'gshen_sqlserver_ct';
SET VAR schema_name = 'default';

SELECT 
  ${catalog_name} as catalog,
  ${schema_name} as schema,
  concat(${catalog_name}, '.', ${schema_name}, '.gateway_logs') as gateway_table,
  concat(${catalog_name}, '.', ${schema_name}, '.pipeline_logs') as pipeline_table

## Step 1: Create Parsed Gateway Logs View

Parse JSON fields and extract table identifiers from flow_name

In [ ]:
%sql
CREATE OR REPLACE VIEW ${catalog_name}.${schema_name}.gateway_logs_parsed AS
SELECT
    -- Identifiers
    g.id as gateway_event_id,
    g.timestamp,
    g.update_id as run_id,
    g.pipeline_id,
    g.pipeline_name,
    
    -- Origin metadata
    g.origin:cloud as cloud,
    g.origin:region as region,
    g.origin:org_id as org_id,
    g.origin:cluster_id as cluster_id,
    
    -- Extract table identifier from flow_name
    -- flow_name format: 'gshen_sqlserver_ct.database_dbo_loadtest_pipeline.table_207'
    g.flow_name,
    regexp_extract(g.flow_name, '([^\\.]+)$', 1) as table_identifier,
    
    -- Event classification
    g.event_type,
    g.level,
    g.maturity_level,
    g.message,
    
    -- Parse flow_progress metrics
    g.details:flow_progress.status as flow_status,
    g.details:flow_progress.metrics.executor_time_ms as executor_time_ms,
    g.details:flow_progress.metrics.executor_cpu_time_ms as executor_cpu_time_ms,
    
    -- Parse stream_progress (nested JSON string)
    g.details:stream_progress.progress_json as stream_progress_raw,
    
    -- Parse background operations (tombstone GC, etc.)
    g.details:background_operation.op_type as bg_op_type,
    g.details:background_operation.status as bg_op_status,
    g.details:background_operation.duration_ms as bg_op_duration_ms,
    g.details:background_operation.num_rows_deleted as bg_rows_deleted,
    g.details:background_operation.num_bytes_deleted as bg_bytes_deleted,
    g.details:background_operation.num_files_deleted as bg_files_deleted
FROM ${catalog_name}.${schema_name}.gateway_logs g
WHERE g.flow_name IS NOT NULL

## Step 2: Create Parsed Pipeline Logs View

Standardize table identifiers for joining

In [ ]:
%sql
CREATE OR REPLACE VIEW ${catalog_name}.${schema_name}.pipeline_logs_parsed AS
SELECT
    -- Run identifiers
    p.run_id,
    p.pipeline_id as ct_pipeline_id,
    p.pipeline_config,
    p.run_started_at,
    p.run_completed_at,
    p.run_duration_seconds,
    p.run_status as overall_run_status,
    p.sync_date,
    
    -- Source connection info
    p.server as source_server,
    p.database as source_database,
    
    -- Table info
    p.table_fqn,
    p.table_name,
    p.schema_name,
    p.uc_catalog,
    p.ingest_pipeline,
    
    -- Extract comparable table identifier
    regexp_extract(p.table_name, '(.+)$', 1) as table_identifier,
    
    -- Sync metrics
    p.rows_written,
    p.files_written,
    p.table_duration_seconds,
    p.table_status,
    
    -- Sync configuration
    p.configured_mode,
    p.actual_mode,
    p.scd_type,
    p.soft_delete,
    
    -- Change tracking info
    p.ct_since_version,
    p.ct_current_version,
    
    -- Schema info
    p.schema_version,
    p.schema_changed,
    p.column_count,
    
    -- Error info
    p.error_message,
    p.error_type,
    p.skip_reason
FROM ${catalog_name}.${schema_name}.pipeline_logs p

## Step 3: Create Unified View (Gateway + Pipeline)

Full join combining both data sources

In [ ]:
%sql
CREATE OR REPLACE VIEW ${catalog_name}.${schema_name}.unified_sync_metrics AS
WITH gateway_parsed AS (
    SELECT
        g.update_id as run_id,
        regexp_extract(g.flow_name, '([^\\.]+)$', 1) as table_identifier,
        g.timestamp as gateway_timestamp,
        g.event_type,
        g.level,
        g.details:flow_progress.metrics.executor_time_ms as executor_time_ms,
        g.details:flow_progress.metrics.executor_cpu_time_ms as executor_cpu_time_ms,
        g.details:flow_progress.status as flow_status,
        g.details:background_operation.op_type as bg_op_type,
        g.details:background_operation.duration_ms as bg_op_duration_ms,
        g.pipeline_name,
        g.origin:cluster_id as cluster_id,
        g.message as gateway_message
    FROM ${catalog_name}.${schema_name}.gateway_logs g
    WHERE g.flow_name IS NOT NULL
),
pipeline_parsed AS (
    SELECT
        p.run_id,
        regexp_extract(p.table_name, '(.+)$', 1) as table_identifier,
        p.table_fqn,
        p.table_name,
        p.schema_name,
        p.source_database as database_name,
        p.uc_catalog,
        p.rows_written,
        p.files_written,
        p.table_duration_seconds,
        p.table_status,
        p.configured_mode,
        p.actual_mode,
        p.scd_type,
        p.ct_current_version,
        p.error_message,
        p.run_started_at,
        p.run_completed_at,
        p.overall_run_status
    FROM ${catalog_name}.${schema_name}.pipeline_logs_parsed p
)
SELECT
    -- Primary identifiers
    COALESCE(g.run_id, p.run_id) as run_id,
    COALESCE(g.table_identifier, p.table_identifier) as table_identifier,
    p.table_fqn,
    p.table_name,
    
    -- Timestamps
    g.gateway_timestamp,
    p.run_started_at,
    p.run_completed_at,
    
    -- Status
    p.overall_run_status,
    p.table_status,
    g.flow_status,
    g.event_type as gateway_event_type,
    g.level as gateway_log_level,
    
    -- Gateway performance metrics
    g.executor_time_ms,
    g.executor_cpu_time_ms,
    CASE 
        WHEN g.executor_time_ms IS NOT NULL THEN g.executor_time_ms / 1000.0 
        ELSE NULL 
    END as executor_time_seconds,
    
    -- Pipeline performance metrics
    p.table_duration_seconds as sync_duration_seconds,
    p.rows_written,
    p.files_written,
    CASE
        WHEN p.table_duration_seconds > 0 AND p.rows_written > 0
        THEN p.rows_written / p.table_duration_seconds
        ELSE NULL
    END as rows_per_second,
    
    -- Configuration
    p.configured_mode,
    p.actual_mode,
    p.scd_type,
    p.ct_current_version as change_tracking_version,
    
    -- Background operations
    g.bg_op_type,
    g.bg_op_duration_ms,
    
    -- Errors
    p.error_message,
    g.gateway_message,
    
    -- Source info
    p.database_name,
    p.schema_name,
    p.uc_catalog,
    g.cluster_id,
    
    -- Data quality flags
    CASE WHEN g.run_id IS NULL THEN true ELSE false END as missing_gateway_data,
    CASE WHEN p.run_id IS NULL THEN true ELSE false END as missing_pipeline_data,
    CASE
        WHEN g.executor_time_ms IS NOT NULL AND p.table_duration_seconds IS NOT NULL
        THEN ABS((g.executor_time_ms / 1000.0) - p.table_duration_seconds)
        ELSE NULL
    END as timing_variance_seconds
FROM gateway_parsed g
FULL OUTER JOIN pipeline_parsed p
    ON g.run_id = p.run_id AND g.table_identifier = p.table_identifier
ORDER BY COALESCE(p.run_started_at, g.gateway_timestamp) DESC, p.table_fqn

## Step 4: Create Materialized View for Performance

For large datasets, use a materialized view with automatic refresh

In [ ]:
%sql
-- Note: Materialized views require appropriate permissions and cluster support
-- Uncomment and modify the schedule as needed

/*
CREATE MATERIALIZED VIEW ${catalog_name}.${schema_name}.unified_sync_metrics_mv
SCHEDULE EVERY 1 HOUR
AS
SELECT * FROM ${catalog_name}.${schema_name}.unified_sync_metrics
WHERE gateway_timestamp > now() - INTERVAL 7 DAYS
   OR run_started_at > now() - INTERVAL 7 DAYS
*/

SELECT 'Materialized view commented out - uncomment and adjust schedule as needed' as note

## Step 5: Example Queries

Common analysis patterns using the unified model

In [ ]:
%sql
-- Query 1: Compare Gateway vs Pipeline timing for each table
SELECT
    table_fqn,
    run_id,
    sync_duration_seconds as pipeline_reported_seconds,
    executor_time_seconds as gateway_reported_seconds,
    timing_variance_seconds,
    rows_written
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
WHERE table_status = 'success'
  AND timing_variance_seconds IS NOT NULL
ORDER BY timing_variance_seconds DESC
LIMIT 20

In [ ]:
%sql
-- Query 2: Tables with high CPU time per row (performance outliers)
SELECT
    table_fqn,
    AVG(executor_cpu_time_ms) as avg_cpu_ms,
    AVG(rows_written) as avg_rows,
    AVG(executor_cpu_time_ms) / NULLIF(AVG(rows_written), 0) as cpu_ms_per_row,
    COUNT(*) as sync_count
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
WHERE gateway_event_type = 'flow_progress'
GROUP BY table_fqn
HAVING COUNT(*) > 1
ORDER BY cpu_ms_per_row DESC

In [ ]:
%sql
-- Query 3: Pipeline runs with incomplete gateway logging
SELECT
    run_id,
    COUNT(*) as pipeline_table_count,
    COUNT(CASE WHEN NOT missing_gateway_data THEN 1 END) as gateway_event_count,
    COUNT(CASE WHEN missing_gateway_data THEN 1 END) as missing_gateway_count
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
GROUP BY run_id
HAVING COUNT(*) != COUNT(CASE WHEN NOT missing_gateway_data THEN 1 END)
ORDER BY missing_gateway_count DESC

In [ ]:
%sql
-- Query 4: Sync throughput by table (rows per second)
SELECT
    table_fqn,
    configured_mode,
    actual_mode,
    AVG(rows_written) as avg_rows,
    AVG(sync_duration_seconds) as avg_duration,
    AVG(rows_per_second) as avg_throughput_rps,
    MAX(rows_per_second) as peak_throughput_rps
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
WHERE table_status = 'success'
  AND rows_written > 0
GROUP BY table_fqn, configured_mode, actual_mode
ORDER BY avg_throughput_rps DESC

In [ ]:
%sql
-- Query 5: Background operations (garbage collection) analysis
SELECT
    table_fqn,
    bg_op_type,
    COUNT(*) as op_count,
    AVG(bg_op_duration_ms) as avg_duration_ms,
    MAX(bg_op_duration_ms) as max_duration_ms
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
WHERE bg_op_type IS NOT NULL
GROUP BY table_fqn, bg_op_type
ORDER BY avg_duration_ms DESC

## Step 6: Create Summary Aggregations Table

For dashboarding and long-term trend analysis

In [ ]:
%sql
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.sync_metrics_summary AS
SELECT
    date_trunc('day', COALESCE(run_started_at, gateway_timestamp)) as sync_date,
    table_fqn,
    uc_catalog,
    database_name,
    schema_name,
    configured_mode,
    scd_type,
    
    -- Count metrics
    COUNT(*) as total_sync_attempts,
    COUNT(CASE WHEN table_status = 'success' THEN 1 END) as successful_syncs,
    COUNT(CASE WHEN table_status != 'success' OR table_status IS NULL THEN 1 END) as failed_syncs,
    
    -- Volume metrics
    SUM(rows_written) as total_rows_written,
    SUM(files_written) as total_files_written,
    AVG(rows_written) as avg_rows_per_sync,
    
    -- Timing metrics from pipeline
    AVG(sync_duration_seconds) as avg_sync_duration,
    MAX(sync_duration_seconds) as max_sync_duration,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sync_duration_seconds) as median_sync_duration,
    
    -- Timing metrics from gateway
    AVG(executor_time_seconds) as avg_executor_time,
    AVG(executor_cpu_time_ms / 1000.0) as avg_cpu_time_seconds,
    
    -- Throughput
    AVG(rows_per_second) as avg_throughput_rps,
    MAX(rows_per_second) as peak_throughput_rps,
    
    -- Background operations
    COUNT(CASE WHEN bg_op_type IS NOT NULL THEN 1 END) as bg_op_count,
    AVG(CASE WHEN bg_op_type IS NOT NULL THEN bg_op_duration_ms END) as avg_bg_op_duration_ms
FROM ${catalog_name}.${schema_name}.unified_sync_metrics
GROUP BY 1, 2, 3, 4, 5, 6, 7

## Summary

Created views:
- `gateway_logs_parsed`: Parsed and enriched gateway logs
- `pipeline_logs_parsed`: Parsed pipeline logs with extracted identifiers
- `unified_sync_metrics`: Combined view joining both sources
- `sync_metrics_summary`: Daily aggregations for trending

Key join logic:
- `run_id` (pipeline) = `update_id` (gateway)
- `table_name` (pipeline) extracted to match `flow_name` suffix (gateway)